# Session 19: Emerging Trends and Upcoming Technologies in AI & ML

**Applied Machine Learning Using Python**  
**Duration**: 4 Hours (TL19)  
**Level**: 🔴 Advanced

---

## 📋 Objectives
1. Implement Explainable AI (XAI) using LIME on a real healthcare model.
2. Simulate Federated Learning with multiple clients and server aggregation.
3. Build a Document Query System using TF-IDF and cosine similarity.
4. Explore the difference between keyword search and semantic search.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
print("✅ Libraries Loaded Successfully.")

---
## §1 — Explainable AI: Why Did the AI Flag This Patient?

You are a Data Scientist at a cardiology clinic. The hospital's AI predicts heart disease risk.
A cardiologist refuses to act on the AI's recommendation unless you can explain **exactly which
medical factors** drove the prediction for each individual patient.

We'll train a complex Gradient Boosting model (a Black Box), then use feature importance to
understand what it learned.

In [ ]:
# Generate synthetic heart disease data
np.random.seed(42)
n = 1500

age = np.random.normal(55, 12, n).clip(25, 85).astype(int)
cholesterol = np.random.normal(240, 45, n).clip(120, 400).astype(int)
max_heart_rate = (220 - age - np.random.normal(0, 15, n)).clip(70, 210).astype(int)
exercise_angina = np.random.binomial(1, 0.3, n)
st_depression = np.random.exponential(1.0, n).clip(0, 6).round(1)
num_vessels = np.random.choice([0, 1, 2, 3], n, p=[0.55, 0.25, 0.13, 0.07])

risk = (0.03*age + 0.005*cholesterol - 0.02*max_heart_rate + 
        0.8*exercise_angina + 0.5*st_depression + 0.7*num_vessels + 
        np.random.normal(0, 0.5, n))
risk_prob = 1 / (1 + np.exp(-(risk - np.median(risk))))
heart_disease = np.random.binomial(1, risk_prob)

df_heart = pd.DataFrame({
    'Age': age, 'Cholesterol': cholesterol, 'MaxHeartRate': max_heart_rate,
    'ExerciseAngina': exercise_angina, 'ST_Depression': st_depression,
    'NumVessels': num_vessels, 'HeartDisease': heart_disease
})

print(f"Dataset: {len(df_heart)} patients, {df_heart['HeartDisease'].mean():.1%} positive")
display(df_heart.head())

In [ ]:
# Train the Black-Box model
features = ['Age', 'Cholesterol', 'MaxHeartRate', 'ExerciseAngina', 'ST_Depression', 'NumVessels']
X = df_heart[features].values
y = df_heart['HeartDisease'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

model = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"✅ Black-Box Model Accuracy: {acc*100:.1f}%")
print(f"⚠️  This model has 150 boosted trees — no human can read the math.")

In [ ]:
# Feature Importance — GLOBAL view of what the model learned
importances = model.feature_importances_
sorted_idx = np.argsort(importances)

plt.figure(figsize=(8, 5))
plt.barh([features[i] for i in sorted_idx], importances[sorted_idx], color='#e74c3c')
plt.xlabel('Feature Importance (Gini)')
plt.title('Heart Disease Model — Global Feature Importance')
plt.tight_layout()
plt.show()

print("\nInterpretation: ST_Depression and ExerciseAngina are the strongest predictors.")
print("But this is GLOBAL — it doesn't explain individual patients. For that, we need LIME.")

---
## §2 — Federated Learning: Training Without Sharing Data

**Scenario**: Three hospitals want to build a joint diabetes predictor.
Due to HIPAA/GDPR, NO patient data can leave any hospital.

We simulate Federated Learning: each hospital trains locally, sends ONLY
its model weights to a central server, and the server averages them.

In [ ]:
# Generate 3 independent hospital datasets
def make_hospital_data(n, seed, bias=0):
    np.random.seed(seed)
    age = np.random.normal(50+bias, 12, n).clip(20, 80)
    bmi = np.random.normal(28+bias*0.3, 6, n).clip(15, 50)
    glucose = np.random.normal(120+bias*2, 35, n).clip(50, 300)
    risk = 0.03*age + 0.05*bmi + 0.015*glucose + np.random.normal(0, 0.8, n)
    diabetes = np.random.binomial(1, 1/(1+np.exp(-(risk-np.median(risk)))))
    return pd.DataFrame({'Age': age, 'BMI': bmi, 'Glucose': glucose, 'Diabetes': diabetes})

hosp_a = make_hospital_data(400, 10, bias=0)
hosp_b = make_hospital_data(300, 20, bias=2)
hosp_c = make_hospital_data(250, 30, bias=-1)

# Global test set (neutral)
test_data = make_hospital_data(500, 999, bias=0.5)
X_test_g = test_data[['Age', 'BMI', 'Glucose']].values
y_test_g = test_data['Diabetes'].values

print(f"Hospital A: {len(hosp_a)} patients | Hospital B: {len(hosp_b)} | Hospital C: {len(hosp_c)}")
print(f"Test Set: {len(test_data)} patients")

In [ ]:
# BASELINE: Each hospital trains alone
isolated_scores = {}
for name, data in [('A', hosp_a), ('B', hosp_b), ('C', hosp_c)]:
    m = LogisticRegression(max_iter=200, random_state=42)
    m.fit(data[['Age', 'BMI', 'Glucose']].values, data['Diabetes'].values)
    score = accuracy_score(y_test_g, m.predict(X_test_g))
    isolated_scores[name] = score
    print(f"Hospital {name} (Isolated): {score*100:.1f}%")

# CENTRALIZED: Pool all data (ILLEGAL!)
all_data = pd.concat([hosp_a, hosp_b, hosp_c])
cm = LogisticRegression(max_iter=200, random_state=42)
cm.fit(all_data[['Age', 'BMI', 'Glucose']].values, all_data['Diabetes'].values)
central_score = accuracy_score(y_test_g, cm.predict(X_test_g))
print(f"\n🚨 Centralized (ALL pooled — ILLEGAL): {central_score*100:.1f}%")

In [ ]:
# FEDERATED LEARNING — Privacy-Preserving Collaboration
features_fl = ['Age', 'BMI', 'Glucose']
n_features = len(features_fl)

global_weights = np.zeros(n_features)
global_bias = 0.0

hospitals = [
    ('A', hosp_a[features_fl].values, hosp_a['Diabetes'].values),
    ('B', hosp_b[features_fl].values, hosp_b['Diabetes'].values),
    ('C', hosp_c[features_fl].values, hosp_c['Diabetes'].values),
]

fed_scores = []
for round_num in range(1, 6):
    client_weights = []
    client_biases = []
    client_sizes = []
    
    for name, X_local, y_local in hospitals:
        local_model = LogisticRegression(max_iter=200, random_state=42)
        local_model.fit(X_local, y_local)
        client_weights.append(local_model.coef_.flatten())
        client_biases.append(local_model.intercept_[0])
        client_sizes.append(len(X_local))
    
    # FedAvg aggregation
    total = sum(client_sizes)
    global_weights = sum(w * (s/total) for w, s in zip(client_weights, client_sizes))
    global_bias = sum(b * (s/total) for b, s in zip(client_biases, client_sizes))
    
    # Evaluate
    eval_model = LogisticRegression()
    eval_model.fit(X_test_g[:2], y_test_g[:2])  # dummy fit
    eval_model.coef_ = global_weights.reshape(1, -1)
    eval_model.intercept_ = np.array([global_bias])
    fed_score = accuracy_score(y_test_g, eval_model.predict(X_test_g))
    fed_scores.append(fed_score)
    print(f"Federated Round {round_num}: Global Accuracy = {fed_score*100:.1f}%")

print(f"\n✅ Final Federated: {fed_scores[-1]*100:.1f}% (vs Centralized: {central_score*100:.1f}%)")
print("No patient data was shared! Only model weights were transmitted.")

In [ ]:
# Visualize the comparison
labels = [f'Hosp {k}\n(Isolated)' for k in isolated_scores] + ['Centralized\n(ILLEGAL)', 'Federated\n(LEGAL)']
scores = list(isolated_scores.values()) + [central_score, fed_scores[-1]]
colors = ['#3498db', '#3498db', '#3498db', '#e74c3c', '#2ecc71']

plt.figure(figsize=(10, 5))
bars = plt.bar(labels, [s*100 for s in scores], color=colors, edgecolor='white', linewidth=2)
plt.ylabel('Accuracy (%)')
plt.title('Federated Learning vs Isolated vs Centralized Training')
plt.ylim(40, 100)

for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{score*100:.1f}%', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## §3 — Document Query System (TF-IDF)

**Scenario**: You are building an internal knowledge search engine.
Employees type natural language questions, and the system finds the
most relevant document using TF-IDF vectorization and cosine similarity.

This is the foundational building block of **RAG (Retrieval-Augmented Generation)**.

In [ ]:
# Mini knowledge base
documents = {
    'Leave Policy': 'Employees are entitled to 21 days paid annual leave. Sick leave requires a medical certificate after 2 days.',
    'Password Policy': 'Passwords must be 12 characters with MFA enabled. Change every 90 days. Never share credentials.',
    'Expense Policy': 'Business expenses must be submitted within 30 days. Receipts required for claims above $50.',
    'Data Protection': 'Customer data is classified as Confidential. GDPR compliance required. Retain records for 7 years.',
    'Remote Work Policy': 'Employees may work remotely 3 days per week. Core hours 10AM-2PM. 25Mbps internet required.',
}

doc_names = list(documents.keys())
doc_texts = list(documents.values())

# Build TF-IDF index
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(doc_texts)

print(f"📚 Indexed {len(documents)} documents")
print(f"   Vocabulary: {len(vectorizer.get_feature_names_out())} terms")
print(f"   Matrix shape: {tfidf_matrix.shape}")

In [ ]:
# Query the knowledge base
queries = [
    "How many vacation days do I get?",
    "How do I set up two-factor authentication?",
    "Can I work from home on Fridays?",
    "What are the rules about storing customer information?",
    "How do I get reimbursed for a flight?"
]

for query in queries:
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, tfidf_matrix).flatten()
    best_idx = sims.argmax()
    
    print(f"\n❓ Query: \"{query}\"")
    print(f"   🥇 Best Match: {doc_names[best_idx]} (Confidence: {sims[best_idx]*100:.1f}%)")
    print(f"   📄 Content: {doc_texts[best_idx][:100]}...")

---
## §4 — Key Emerging Trends Summary

| Trend | What It Is | Why It Matters |
|-------|-----------|----------------|
| **Foundation Models** | Massive pre-trained models fine-tuned for specific tasks | One model, many applications |
| **Edge AI** | Running AI on devices (phones, drones, cars) | Zero latency, works offline |
| **Generative AI** | AI that creates (text, images, molecules, chip designs) | Accelerates R&D 10-100x |
| **AI Regulation** | EU AI Act, NDPA, Executive Orders | Legal compliance now mandatory |
| **MLOps** | DevOps for ML (monitoring, retraining, CI/CD) | Models decay without maintenance |
| **Multimodal AI** | Models processing text + images + audio together | Richer, more robust understanding |
| **AI Agents** | Autonomous task execution using tools and APIs | From assistants to autonomous workers |
| **RAG** | Grounding LLMs in retrieved documents | Eliminates hallucinations |

## Conclusion

The AI landscape is shifting from **"Can we build it?"** to **"Should we build it, and can we explain it?"**

The technologies in this session — XAI, Federated Learning, RAG, and Edge AI — are not theoretical. They are in production at Google, Apple, NVIDIA, and across the healthcare and financial industries. As an ML engineer, your competitive advantage lies in understanding not just *how* to build models, but how to build them **responsibly, privately, and transparently**.